# 🚩 शिव एआई (Shiv AI) - सेल्फ-लर्निंग फैक्ट्री
**मालिक:** श्री राम नाग  
यह नोटबुक विकिपीडिया, गिटहब और हगिंग फेस से डेटा लेकर खुद को ट्रेन करती है।

In [ ]:
# @title ⚙️ सेटअप और लर्निंग टूल्स (टर्बो स्पीड)
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q --no-deps "xformers<0.0.27" "trl<0.9.0" peft accelerate bitsandbytes
!pip install -q wikipedia requests datasets
print("✅ शिव एआई के सीखने के औजार तैयार हैं!")

In [ ]:
# @title 🌐 इंटरनेट से महाज्ञान डाउनलोड करना
import wikipedia
import requests
from datasets import Dataset

def get_knowledge():
    wikipedia.set_lang("hi")
    kb = []
    # १. विकिपीडिया ज्ञान
    topics = ["मशीन लर्निंग", "आर्टिफिशियल इंटेलिजेंस", "ब्रह्मांड"]
    for t in topics:
        try:
            p = wikipedia.page(t)
            kb.append({"instruction": f"{t} क्या है?", "response": p.summary})
        except: pass
    
    # २. गिटहब से कोडिंग ज्ञान (आपका अपना app.py)
    repo_url = "https://raw.githubusercontent.com/shriramnag/Shiv-AI/main/app.py"
    code = requests.get(repo_url).text
    kb.append({"instruction": "शिव एआई का app.py कोड समझाएं", "response": code[:1000]})
    
    return kb

training_data = get_knowledge()
dataset = Dataset.from_list(training_data)
print(f"✅ {len(training_data)} नए विषय इंटरनेट से लोड हो गए!")

In [ ]:
# @title 🏗️ ट्रेनिंग शुरू करें (टर्बो इंजन)
from unsloth import FastLanguageModel
from trl import SFTTrainer
from transformers import TrainingArguments

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "HuggingFaceTB/SmolLM2-135M-Instruct",
    load_in_4bit = True,
)

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "instruction",
    args = TrainingArguments(
        max_steps = 50,
        learning_rate = 2e-4,
        output_dir = "ShivAI_Knowledge",
    ),
)
trainer.train()
print("✅ शिव एआई अब और भी ज्ञानी हो गया है!")